# NYC Mobility - Great Expectations Quality Gate

This notebook is the final end-to-end quality gate for the NYC Mobility pipeline.

It evaluates 14 governed data-quality checks across:

- Consistency
- Completeness
- Uniqueness
- Validity
- Timeliness and volume
- Accuracy scope
- Auditability

The data-quality checks are calculated using PySpark.

The results are published to the Gold layer for the Data Quality Dashboard, then Great Expectations validates that:

- exactly 14 governed checks are present;
- check names are present and unique;
- quality attributes are present;
- statuses are present; and
- every governed check has a `PASS` status.

The Databricks job fails if any Great Expectations validation fails.

## Python Imports and Great Expectations Setup

Great Expectations is configured as the final validation layer for the governed quality checks.

The GX dependency is managed by the Databricks job configuration rather than installed inside this notebook.

In [0]:
import great_expectations as gx
from pyspark.sql import functions as F

# No persistent GX project is needed for this pipeline run.
context = gx.get_context(mode="ephemeral")

## Create Quality Check Results

Each governed check records:

- `check_name`: name of the validation
- `quality_attribute`: quality dimension being evaluated
- `expected_value`: expected condition
- `actual_value`: calculated result
- `status`: `PASS` or `FAIL`

The checks use PySpark and existing Gold-layer validation objects to evaluate the pipeline.

A total of 14 governed checks are produced before Great Expectations runs.

Each check produces an expected value, actual value, and `PASS` or `FAIL` result.

The complete set must contain exactly 14 governed checks before the final Great Expectations validation can pass.

In [0]:
bronze_green = spark.table("`ftw-week-08`.`01_bronze`.green_taxi")
silver_green = spark.table("`ftw-week-08`.`02_silver`.green_taxi")
gold_fact = spark.table("`ftw-week-08`.`03_gold`.fact_green_taxi_trip")
silver_zones = spark.table("`ftw-week-08`.`02_silver`.taxi_zones")
gold_zone = spark.table("`ftw-week-08`.`03_gold`.dim_taxi_zone")
silver_weather = spark.table("`ftw-week-08`.`02_silver`.weather")
gold_weather = spark.table("`ftw-week-08`.`03_gold`.dim_weather_hour")
gold_date = spark.table("`ftw-week-08`.`03_gold`.dim_date")
gold_time = spark.table("`ftw-week-08`.`03_gold`.dim_time")
dq_referential = spark.table(
    "`ftw-week-08`.`03_gold`.dq_dashboard_referential_integrity"
)
analytics_validation = spark.table(
    "`ftw-week-08`.`03_gold`.analytics_validation_summary"
)
dq_dimensions = spark.table(
    "`ftw-week-08`.`03_gold`.dq_dashboard_canonical_dimensions"
)

checks = []

# 1. Bronze to Silver Green Taxi row preservation

bronze_count = bronze_green.count()
silver_count = silver_green.count()

checks.append((
    "Bronze to Silver Green Taxi row preservation",
    "CONSISTENCY",
    str(bronze_count),
    str(silver_count),
    "PASS" if bronze_count == silver_count else "FAIL"
))


# 2. Silver to Gold fact row preservation

gold_fact_count = gold_fact.count()

checks.append((
    "Silver to Gold fact row preservation",
    "CONSISTENCY",
    str(silver_count),
    str(gold_fact_count),
    "PASS" if silver_count == gold_fact_count else "FAIL"
))


# 3. Silver to Gold retained measures reconcile

measure_columns = [
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "cbd_congestion_fee",
    "total_amount"
]

silver_measures = silver_green.select([
    F.sum(F.col(c)).alias(c)
    for c in measure_columns
]).first()

gold_measures = gold_fact.select([
    F.sum(F.col(c)).alias(c)
    for c in measure_columns
]).first()

measure_difference = sum(
    abs(
        float(silver_measures[c] or 0)
        - float(gold_measures[c] or 0)
    )
    for c in measure_columns
)

MEASURE_TOLERANCE = 1e-5

display_difference = round(measure_difference, 6)

checks.append((
    "Silver to Gold retained measures reconcile",
    "CONSISTENCY",
    f"aggregate difference <= {MEASURE_TOLERANCE}",
    f"{display_difference} aggregate difference",
    "PASS" if measure_difference <= MEASURE_TOLERANCE else "FAIL"
))


# 4. Silver to Gold Taxi Zone row preservation

silver_zone_count = silver_zones.count()
gold_zone_count = gold_zone.count()

checks.append((
    "Silver to Gold Taxi Zone row preservation",
    "CONSISTENCY",
    str(silver_zone_count),
    str(gold_zone_count),
    "PASS" if silver_zone_count == gold_zone_count else "FAIL"
))


# 5. Gold Weather contains Silver hours plus one Unknown

silver_weather_count = silver_weather.count()
gold_weather_count = gold_weather.count()

expected_weather_count = silver_weather_count + 1

checks.append((
    "Gold Weather contains Silver hours plus one Unknown member",
    "COMPLETENESS",
    str(expected_weather_count),
    str(gold_weather_count),
    "PASS" if gold_weather_count == expected_weather_count else "FAIL"
))


# 6. Gold dimension keys are unique

null_date_keys = gold_date.filter(F.col("date_key").isNull()).count()
null_time_keys = gold_time.filter(F.col("time_key").isNull()).count()
null_zone_keys = gold_zone.filter(F.col("taxi_zone_key").isNull()).count()
null_weather_keys = gold_weather.filter(F.col("weather_hour_key").isNull()).count()

dup_date = (
    gold_date.filter(F.col("date_key").isNotNull()).count()
    - gold_date.filter(F.col("date_key").isNotNull()).select(F.countDistinct("date_key")).first()[0]
)
dup_time = (
    gold_time.filter(F.col("time_key").isNotNull()).count()
    - gold_time.filter(F.col("time_key").isNotNull()).select(F.countDistinct("time_key")).first()[0]
)
dup_zone = (
    gold_zone.filter(F.col("taxi_zone_key").isNotNull()).count()
    - gold_zone.filter(F.col("taxi_zone_key").isNotNull()).select(F.countDistinct("taxi_zone_key")).first()[0]
)
dup_weather = (
    gold_weather.filter(F.col("weather_hour_key").isNotNull()).count()
    - gold_weather.filter(F.col("weather_hour_key").isNotNull()).select(F.countDistinct("weather_hour_key")).first()[0]
)

total_null_keys = null_date_keys + null_time_keys + null_zone_keys + null_weather_keys
total_dup_keys = dup_date + dup_time + dup_zone + dup_weather
total_key_issues = total_null_keys + total_dup_keys

checks.append((
    "Gold dimension keys are unique",
    "UNIQUENESS",
    "0 duplicate or null-key rows",
    str(total_key_issues),
    "PASS" if total_key_issues == 0 else "FAIL"
))


# 7. Gold fact technical keys are complete and unique

fact_key_stats = gold_fact.agg(
    F.count_if(
        F.col("trip_key").isNull()
    ).alias("null_keys"),
    (
        F.count("trip_key")
        - F.countDistinct("trip_key")
    ).alias("dup_keys")
).first()

total_trip_key_issues = (
    fact_key_stats["null_keys"]
    + fact_key_stats["dup_keys"]
)

checks.append((
    "Gold fact technical keys are complete and unique",
    "UNIQUENESS",
    "0 null or duplicate-key rows",
    str(total_trip_key_issues),
    "PASS" if total_trip_key_issues == 0 else "FAIL"
))

# 8. Fact foreign keys resolve without join multiplication

ri_row = dq_referential.first()
if ri_row is None:
    checks.append((
        "Fact foreign keys resolve without join multiplication",
        "CONSISTENCY",
        "0 missing references and 0 added rows",
        "dq_dashboard_referential_integrity view returned 0 rows",
        "FAIL"
    ))

else:
    total_missing = (
        int(ri_row["missing_pickup_date_keys"] or 0)
        + int(ri_row["missing_dropoff_date_keys"] or 0)
        + int(ri_row["missing_pickup_time_keys"] or 0)
        + int(ri_row["missing_dropoff_time_keys"] or 0)
        + int(ri_row["missing_pickup_zone_keys"] or 0)
        + int(ri_row["missing_dropoff_zone_keys"] or 0)
        + int(ri_row["missing_weather_keys"] or 0)
    )
    join_diff = abs(int(ri_row["join_row_difference"] or 0))
    ri_total = total_missing + join_diff

    checks.append((
        "Fact foreign keys resolve without join multiplication",
        "CONSISTENCY",
        "0 missing references and 0 added rows",
        str(ri_total),
        "PASS" if ri_total == 0 else "FAIL"
    ))


# 9. Gold fact lineage is complete

missing_lineage = gold_fact.filter(
    F.col("source_system").isNull()
    | F.col("source_file").isNull()
    | F.col("batch_id").isNull()
).count()

checks.append((
    "Gold fact lineage is complete",
    "AUDITABILITY",
    "0 rows missing lineage",
    str(missing_lineage),
    "PASS" if missing_lineage == 0 else "FAIL"
))


# 10. Gold fact quality flags are populated

null_quality_flags = gold_fact.filter(
    F.col("dq_zero_trip_distance").isNull()
    | F.col("dq_extreme_trip_distance").isNull()
    | F.col("dq_negative_trip_distance").isNull()
    | F.col("dq_out_of_range_datetime").isNull()
    | F.col("dq_invalid_trip_duration").isNull()
    | F.col("dq_missing_weather_coverage").isNull()
).count()

checks.append((
    "Gold fact quality flags are populated",
    "VALIDITY",
    "0 null quality flags",
    str(null_quality_flags),
    "PASS" if null_quality_flags == 0 else "FAIL"
))


# 11. Silver Weather hourly sequence is continuous

weather_bounds = silver_weather.agg(
    F.countDistinct("weather_datetime").alias("observed_hour_count"),
    F.min("weather_datetime").alias("min_datetime"),
    F.max("weather_datetime").alias("max_datetime")
)

weather_stats = weather_bounds.select(
    "observed_hour_count",
    F.timestamp_diff(
        "HOUR",
        F.col("min_datetime"),
        F.col("max_datetime")
    ).alias("hour_difference")
).first()

observed_hour_count = weather_stats["observed_hour_count"]
expected_hour_count = weather_stats["hour_difference"] + 1

checks.append((
    "Silver Weather hourly sequence is continuous",
    "TIMELINESS_VOLUME",
    f"{expected_hour_count} expected hours",
    f"{observed_hour_count} observed hours",
    "PASS" if observed_hour_count == expected_hour_count else "FAIL"
))


# 12. Accuracy claim is correctly scoped

expected_dimension_keys = {
    "COMPLETENESS",
    "VALIDITY",
    "UNIQUENESS",
    "CONSISTENCY",
    "ACCURACY",
    "AUDITABILITY",
    "TIMELINESS_VOLUME",
}

dimension_rows = {
    row["dimension_key"]: row.asDict()
    for row in dq_dimensions.select(
        "dimension_key",
        "measurement_status",
        "measurement_note",
    ).collect()
}

accuracy_row = dimension_rows.get("ACCURACY")

accuracy_scope_ok = (
    accuracy_row is not None
    and accuracy_row["measurement_status"] == "NOT_MEASURED"
    and "no external ground truth"
    in (accuracy_row["measurement_note"] or "").lower()
)

checks.append((
    "Accuracy claim is correctly scoped",
    "ACCURACY",
    (
        "Accuracy is NOT_MEASURED because "
        "no external ground truth exists"
    ),
    str(accuracy_row),
    "PASS" if accuracy_scope_ok else "FAIL",
))


# 13. Analytics validation passes

row = analytics_validation.select("overall_status").first()
analytics_status = row["overall_status"] if row else "FAIL"

checks.append((
    "Analytics validation passes",
    "VALIDITY",
    "PASS",
    str(analytics_status),
    "PASS" if analytics_status == "PASS" else "FAIL"
))


# 14. All seven quality attributes are documented

actual_dimension_keys = set(dimension_rows)

dimension_keys_ok = (
    actual_dimension_keys == expected_dimension_keys
)

checks.append((
    "All seven quality attributes are documented",
    "AUDITABILITY",
    ", ".join(sorted(expected_dimension_keys)),
    ", ".join(sorted(actual_dimension_keys)),
    "PASS" if dimension_keys_ok else "FAIL",
))

## Review All Checks

The checks above validate all seven quality attributes across 14 governed checks:

1. Bronze to Silver Green Taxi row preservation
2. Silver to Gold fact row preservation
3. Silver to Gold retained measure reconciliation
4. Silver to Gold Taxi Zone row preservation
5. Gold Weather completeness
6. Gold dimension keys are unique (null and duplicate detection)
7. Gold fact technical keys are complete and unique (null and duplicate detection)
8. Fact foreign keys resolve without join multiplication
9. Gold fact lineage is complete
10. Gold fact quality flags are populated
11. Silver Weather hourly sequence is continuous
12. Accuracy claim is correctly scoped
13. Analytics validation passes
14. All seven quality attributes are documented

Each check produces an expected value, actual value, and PASS/FAIL result.

### Display Quality Check Results

In [0]:
checks_df = spark.createDataFrame(
    checks,
    [
        "check_name",
        "quality_attribute",
        "expected_value",
        "actual_value",
        "status"
    ]
)

display(checks_df)

## Publish Quality Gate Results

The calculated quality checks are published to the Gold layer so the Data Quality Dashboard continues to receive the latest validation evidence.

The notebook publishes:

- `pipeline_quality_gate_results_data` — Delta table containing the latest governed check results
- `pipeline_quality_gate_results` — view used to expose the quality-gate results
- `pipeline_quality_gate_summary` — summary of total, passed, and failed checks and the overall quality-gate status

The objects are recreated on each run so repeated pipeline executions do not append duplicate quality-gate rows.

In [0]:
catalog = "ftw-week-08"
schema = "03_gold"

results_data = (
    f"`{catalog}`.`{schema}`."
    "`pipeline_quality_gate_results_data`"
)

results_view = (
    f"`{catalog}`.`{schema}`."
    "`pipeline_quality_gate_results`"
)

summary_view = (
    f"`{catalog}`.`{schema}`."
    "`pipeline_quality_gate_summary`"
)

published_checks_df = checks_df.withColumn(
    "checked_at",
    F.current_timestamp()
)

published_checks_df.createOrReplaceTempView(
    "gx_quality_gate_results_for_publish"
)

spark.sql(f"""
CREATE OR REPLACE TABLE {results_data}
USING DELTA
AS
SELECT
    check_name,
    quality_attribute,
    expected_value,
    actual_value,
    status,
    checked_at
FROM gx_quality_gate_results_for_publish
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {results_view} AS
SELECT
    check_name,
    quality_attribute,
    expected_value,
    actual_value,
    status,
    checked_at
FROM {results_data}
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {summary_view} AS
SELECT
    MAX(checked_at) AS checked_at,
    COUNT(*) AS total_checks,
    SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)
        AS passed_checks,
    SUM(CASE WHEN status <> 'PASS' THEN 1 ELSE 0 END)
        AS failed_checks,
    CASE
        WHEN COUNT(*) = 14
         AND SUM(CASE WHEN status <> 'PASS' THEN 1 ELSE 0 END) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS overall_status
FROM {results_view}
""")

display(spark.table(results_view))
display(spark.table(summary_view))

## Great Expectations Quality Gate

Great Expectations acts as the final validation layer for the 14 governed quality checks.

GX validates that:

- exactly 14 checks are present;
- `check_name` values are not null;
- `check_name` values are unique;
- `quality_attribute` values are not null;
- `status` values are not null; and
- every `status` is `PASS`.

The PySpark checks remain the source of the calculated data-quality results. Great Expectations validates that the complete governed check set is present, structurally valid, and passing.

The checks DataFrame is converted to pandas because it contains only 14 rows and is small enough to collect safely.

In [0]:
# DEMO ONLY: intentionally force one in-memory quality result to fail.
# This branch must never be merged into main.
DEMO_FORCE_GX_FAILURE = False

gx_checks_df = checks_df

if DEMO_FORCE_GX_FAILURE:
    demo_check_name = "Analytics validation passes"

    gx_checks_df = (
        checks_df
        .withColumn(
            "status",
            F.when(
                F.col("check_name") == demo_check_name,
                F.lit("FAIL"),
            ).otherwise(F.col("status")),
        )
        .withColumn(
            "actual_value",
            F.when(
                F.col("check_name") == demo_check_name,
                F.lit("DEMO_ONLY: intentionally forced failure"),
            ).otherwise(F.col("actual_value")),
        )
    )

    print(
        "DEMO MODE: forcing an in-memory GX failure for:",
        demo_check_name,
    )

# Convert only the controlled GX input to pandas (14 rows - safe to collect).
checks_pdf = gx_checks_df.toPandas()

# Create a pandas data source and DataFrame asset
gx_source = context.data_sources.add_or_update_pandas("quality_gate_source")
gx_asset = gx_source.add_dataframe_asset(name="checks_asset")
gx_batch = gx_asset.add_batch_definition_whole_dataframe("checks_batch")

# Build the GX expectation suite for all governed quality checks
gx_suite = context.suites.add_or_update(gx.ExpectationSuite(name="quality_gate_suite"))

# Exactly 14 governed checks must exist
gx_suite.add_expectation(
    gx.expectations.ExpectTableRowCountToEqual(
        value=14
    )
)
# Check names must not be null
gx_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="check_name"
    )
)
# Check names must be unique
gx_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="check_name"
    )
)
# Quality attributes must not be null
gx_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="quality_attribute"
    )
)
# Status must not be null
gx_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="status"
    )
)
# Every governed check must have PASS status
gx_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="status",
        value_set=["PASS"],
        result_format="COMPLETE"
    )
)

# Create and run the validation definition
gx_validation = context.validation_definitions.add_or_update(
    gx.ValidationDefinition(
        name="quality_gate_validation",
        data=gx_batch,
        suite=gx_suite,
    )
)

try:
    gx_result = gx_validation.run(
        batch_parameters={"dataframe": checks_pdf},
        result_format="COMPLETE"
    )

except Exception as e:
    raise Exception(
        f"Great Expectations setup/run error (not a DQ check failure): {e}"
    )


print(f"GX validation success: {gx_result.success}")
print(f"Statistics: {gx_result.statistics}")

## Fail/Pass Pipeline

If all Great Expectations validations succeed, the quality gate passes and the pipeline can continue.

If any GX expectation fails, the notebook reports the failed expectation.

For a failed `status = PASS` expectation, GX's detailed validation result is used to identify the specific governed check names that failed.

The notebook then raises an exception so the Databricks job stops and the pipeline is not considered successful.

In [0]:
if gx_result.success:
    print(
        "✓ Great Expectations quality gate PASSED — "
        "all required checks are valid and passing."
    )

else:
    print("✗ Great Expectations quality gate FAILED.")

    for result in gx_result.results:
        if not result.success:
            print(
                "Failed expectation:",
                result.expectation_config.type
            )

            # If the failed expectation is the status = PASS check, show the actual governed check names that failed
            if (
                result.expectation_config.type
                == "expect_column_values_to_be_in_set"
            ):
                failed_names = checks_pdf.loc[
                    checks_pdf["status"] != "PASS",
                    "check_name"
                ].tolist()

                for name in failed_names:
                    print(f"  - {name}")

    raise Exception(
        "Great Expectations quality gate failed. "
        "Pipeline execution stopped."
    )

## Overall gx workflow:

PySpark checks → 14 governed results → publish dashboard evidence → GX validates structure + PASS statuses → pipeline passes or stops